# Notebook 2 — Entity Normalisation — food, species & location grouping
### RapidFuzz fuzzy matching + domain-informed manual mapping + primary venue extraction

**Goal:** Reduce free-text noise in `Food`, `Species`, and `Location` columns so that
downstream analysis can aggregate meaningfully across 18 years and 55 reporting jurisdictions.

**Input:** `cleaned_data.csv` — 18,828 records (output of `01_data_cleaning.ipynb`)  
**Output:** `cleaned_data_grouped.csv` — same records + three new columns:
`Food_grouped`, `Species_grouped`, `primary_location`

**Why all three columns need normalisation:**

| Column | Raw unique values | Problem type | Method |
|---|---|---|---|
| `Food` | 3,128 | Spelling variants + semantic ambiguity | RapidFuzz (threshold 70) + manual map |
| `Species` | 201 | Nomenclature variants + co-infections | RapidFuzz (threshold 85) + manual map |
| `Location` | 161 | Multi-venue strings ("A; B; C") | Semicolon split → primary venue |

`Location` does not require fuzzy matching — the raw values are already standardised
vocabulary (no spelling variants). The only problem is that multi-venue outbreaks
("Restaurant; Catering Service") are stored as a single string, inflating unique counts
from ~20 meaningful venue types to 161. Extracting the first-listed venue resolves this.

## 1. Setup

In [2]:
import pandas as pd
from rapidfuzz import process, fuzz

df = pd.read_csv("data/cleaned_data.csv")
print(f"Input: {df.shape[0]:,} records × {df.shape[1]} columns")
print(f"Food unique strings:    {df['Food'].nunique():,}")
print(f"Species unique strings: {df['Species'].nunique():,}")
print(f"Location unique strings: {df['Location'].nunique():,}")
print()

# Preview Location complexity
compound_n = df['Location'].str.contains(';', na=False).sum()
print(f"Multi-venue Location records: {compound_n:,} ({compound_n/len(df)*100:.1f}%)")
print("Sample multi-venue strings:")
print(df[df['Location'].str.contains(';', na=False)]['Location']
      .value_counts().head(5).to_string())

Input: 18,828 records × 9 columns
Food unique strings:    3,128
Species unique strings: 201
Location unique strings: 161

Multi-venue Location records: 1,119 (5.9%)
Sample multi-venue strings:
Location
Restaurant; Private Home/Residence          205
Restaurant; Catering Service                163
Private Home/Residence; Grocery Store        97
Private Home/Residence; Catering Service     83
Restaurant; Grocery Store                    59


## 2. Fuzzy matching utility

Reused for both Food and Species — single implementation, two applications.

In [4]:
def fuzzy_group(series: pd.Series, threshold: int) -> dict:
    """
    Build a mapping {original_string: canonical_string} using RapidFuzz
    token_sort_ratio. Strings scoring >= threshold are mapped to the first
    seen variant in that cluster.

    Parameters
    ----------
    series    : pd.Series — column to normalise
    threshold : int — similarity cutoff (0–100). Higher = stricter.
                Food uses 70 (more variant spellings expected).
                Species uses 85 (scientific names are more regular).

    Returns
    -------
    dict mapping each unique string to its canonical form.
    """
    unique_names = series.unique()
    mapping = {}
    for name in unique_names:
        if name in mapping:
            continue
        matches = process.extract(
            name, unique_names,
            scorer=fuzz.token_sort_ratio,
            limit=None
        )
        similar = [m for m, score, _ in matches if score >= threshold]
        for s in similar:
            mapping[s] = name
    return mapping


def apply_manual_map(df: pd.DataFrame, col: str, manual: dict) -> pd.DataFrame:
    """Apply a manual override mapping and print how many values changed."""
    before = df[col].value_counts().to_dict()
    df[col] = df[col].replace(manual)
    changed = sum(1 for k in manual if k in before)
    print(f"  {changed}/{len(manual)} keys matched and replaced")
    return df

## 3. Food grouping

### 3.1 Fuzzy auto-clustering (threshold = 70)

A lower threshold (70) is appropriate for food names because the same food item
appears with highly variable phrasing across reporters and years.

In [20]:
# Normalise whitespace before fuzzy matching
df["Food"] = df["Food"].astype(str).str.strip()

food_mapping = fuzzy_group(df["Food"], threshold=70)
df["Food_grouped"] = df["Food"].map(food_mapping)

print(f"Food unique strings:  {df['Food'].nunique():,}  → after fuzzy: {df['Food_grouped'].nunique():,}")
print()
df["Food_grouped"].value_counts().head(20)

Food unique strings:  3,128  → after fuzzy: 1,420



Food_grouped
Unknown                                          8702
Multiple Foods                                    216
Oysters, Raw                                      191
Beef, Bbq; Ham, Unspecified                       183
Salad, Unspecified                                177
Fruit Punch, Unspecified                          155
Rice; Beans, Unspecified                          152
Ground Beef, Hamburger                            128
Chicken, Teriyaki; Sushi, Unspecified             117
Chocolate Milk, Unspecified                       113
Pie, Unspecified; Mousse                          106
Butter; Carrots, Unspecified                      106
Pasta-Based Salads Unspecified; Chicken Salad     104
Potato Salad                                      101
Chicken, Fried                                     97
Dessert Unspecified                                94
Sandwich, Submarine                                90
Salad                                              89
Duck, Other    

### 3.2 Manual mapping — domain corrections and analytical decisions

Fuzzy matching clusters by string similarity, not biological or culinary meaning.
A second pass of manual corrections is required for cases where string similarity
alone cannot determine the appropriate category.

**Two types of corrections — and an important distinction:**

**Type A — Unambiguous normalisation** (string variants of the same category):
- "Greek Salad", "Caesar Salad", "House Salad" → `Salad`
- "Chicken, Fried", "Chicken, BBQ", "Chicken, Nuggets" → `Chicken`
- "Ground Beef, Hamburger", "Ground Beef, Cheeseburger" → `Ground Beef`

These are straightforward: the food item is unambiguously the same category
regardless of preparation method.

**Type B — Analytical decisions under uncertainty** (multi-ingredient dishes):
- "Chicken Salad" → `Chicken`
- "Potato Salad" → `Salad`
- "Fruit Punch, Unspecified" → `Fruit`

These require an explicit caveat: **grouping multi-ingredient dishes into a
single primary category is an analytical simplification, not a microbiological
determination.**

In "Chicken Salad", the contamination source could be the chicken, the
mayonnaise, the celery, or cross-contamination during preparation.
Without laboratory confirmation — which FDOSS does not require for reporting —
the true vehicle is unknown. The mapping decision (chicken as primary vehicle)
reflects a categorisation choice for aggregation purposes only.

This is a known limitation of food vehicle surveillance data generally:
the reported food is the *epidemiologically implicated vehicle*, not necessarily
the *microbiologically confirmed source*. Type B mappings introduce systematic
ambiguity that should be considered when interpreting food-vehicle rankings,
particularly for mixed/prepared dishes.

In [21]:
FOOD_MANUAL = {
    # Chicken variants
    "Chicken, Fried":                           "Chicken",
    "Chicken Salad":                            "Chicken",
    "Chicken Nachos":                           "Chicken",
    "Chicken, Teriyaki; Sushi, Unspecified":    "Chicken",
    "Chicken, Bbq":                             "Chicken",
    "Chicken; Ribs":                            "Chicken",
    "Chicken, Buffalo Wings":                   "Chicken",
    "Chicken, Nuggets/Fingers":                 "Chicken",
    "Chicken, Confit":                          "Chicken",

    # Fish variants
    "Fish, Ahi":                                "Fish",
    "Fish, Mahi Mahi":                          "Fish",
    "Salmon, Raw; Sushi, Unspecified":          "Fish",
    "Fish, Escolar":                            "Fish",
    "Fish, Baracuda":                           "Fish",
    "Fish, Trout":                              "Fish",
    "Fish, Grouper, Unspecified":               "Fish",
    "Fish, Amberjack":                          "Fish",
    "Fish, Amberjack ":                         "Fish",
    "Unspecified Fish; Applesauce":             "Fish",

    # Salad variants
    "Salad, Unspecified":                       "Salad",
    "Pasta-Based Salads Unspecified; Chicken Salad": "Salad",
    "Potato Salad":                             "Salad",
    "Fruit Salad":                              "Salad",
    "House Salad":                              "Salad",
    "Greek Salad":                              "Salad",
    "Caesar Salad":                             "Salad",

    # Sandwich variants
    "Sandwich, Reuben; Pudding, Unspecified":   "Sandwich",
    "Sandwich, Gyro":                           "Sandwich",
    "Sandwich, Submarine":                      "Sandwich",
    "Sandwich, Turkey; Cantaloupe":             "Sandwich",
    "Sandwich, Cuban":                          "Sandwich",

    # Ground beef
    "Ground Beef, Hamburger":                   "Ground Beef",
    "Ground Beef, Cheeseburger":                "Ground Beef",
    "Ground Beef, Meatballs":                   "Ground Beef",

    # Beef / Pork / Turkey
    "Beef, Bbq; Ham, Unspecified":              "Beef",
    "Beef, Other":                              "Beef",
    "Gravy, Unspecified; Roast Beef, Unspecified; Pork, Roasted": "Beef",
    "Steak, Unspecified; Chicken, Cacciatore; Eggplant, Unspecified": "Steak",
    "Steak, Prime Rib":                         "Steak",
    "Pork, Bbq":                                "Pork",
    "Pork, Roasted":                            "Pork",
    "Pork, Other":                              "Pork",
    "Turkey, Smoked; Ham":                      "Turkey",
    "Turkey, Roasted":                          "Turkey",

    # Dairy / produce / other
    "Fruit Punch, Unspecified":                 "Fruit",
    "Banana, Unspecified; Cantaloupe; Pineapple, Unspecified": "Fruit",
    "Leaf Lettuce, Unspecified":                "Lettuce",
    "Whole Milk, Unpasteurized":                "Milk",
    "Whole Milk, Unpasteurized; Goat Milk, Unpasteurized": "Milk",
    "Chocolate Milk, Unspecified":              "Chocolate Milk",
    "Rice; Beans, Unspecified":                 "Rice/Beans",
    "Rice, Unspecified (Source Unknown)":       "Rice/Beans",
    "Pizza, Meat And Vegetable":                "Pizza",
    "Pizza, Meat":                              "Pizza",
    "Coleslaw; Potato, Unspecified":            "Coleslaw",
    "Cake, Unspecified; Ice Cream, Commercial": "Dessert",
    "Dessert Unspecified":                      "Dessert",
    "Doughnuts, Unspecified":                   "Doughnuts",
    "Butter; Carrots, Unspecified":             "Butter",
    "Pie, Unspecified; Mousse":                 "Pie",
}

print(f"Applying {len(FOOD_MANUAL)} manual corrections...")
df = apply_manual_map(df, "Food_grouped", FOOD_MANUAL)
print(f"Food_grouped unique after manual: {df['Food_grouped'].nunique():,}")
print()
df["Food_grouped"].value_counts().head(15)

Applying 60 manual corrections...
  59/60 keys matched and replaced
Food_grouped unique after manual: 1,366



Food_grouped
Unknown           8702
Salad              677
Chicken            570
Fish               451
Sandwich           309
Ground Beef        273
Beef               266
Multiple Foods     216
Rice/Beans         206
Fruit              205
Pork               191
Oysters, Raw       191
Pizza              136
Dessert            119
Chocolate Milk     113
Name: count, dtype: int64

### 3.3 Food grouping result

In [22]:
food_before = df["Food"].nunique()
food_after  = df["Food_grouped"].nunique()
print(f"Food: {food_before:,} unique strings → {food_after:,} grouped categories")
print(f"Reduction: {(1 - food_after/food_before)*100:.0f}%")
print()
print("Top 10 food groups by outbreak frequency:")
print(df[df["Food_grouped"] != "Unknown"]["Food_grouped"].value_counts().head(10).to_string())

Food: 3,128 unique strings → 1,366 grouped categories
Reduction: 56%

Top 10 food groups by outbreak frequency:
Food_grouped
Salad             677
Chicken           570
Fish              451
Sandwich          309
Ground Beef       273
Beef              266
Multiple Foods    216
Rice/Beans        206
Fruit             205
Oysters, Raw      191


## 4. Species grouping

### 4.1 Fuzzy auto-clustering (threshold = 85)

A higher threshold (85) is used for species names because scientific nomenclature is more
regular — we want to cluster only genuinely similar strings, not merge different species.

In [23]:
df["Species"] = df["Species"].astype(str).str.strip()

species_mapping = fuzzy_group(df["Species"], threshold=85)
df["Species_grouped"] = df["Species"].map(species_mapping)

print(f"Species unique strings: {df['Species'].nunique():,}  → after fuzzy: {df['Species_grouped'].nunique():,}")
print()
df["Species_grouped"].value_counts().head(20)

Species unique strings: 201  → after fuzzy: 155



Species_grouped
Unknown                                     6406
Norovirus genogroup I                       4141
Salmonella enterica                         2299
Norovirus unknown                            777
Clostridium perfringens                      732
Staphylococcus aureus                        531
Escherichia coli, Shiga toxin-producing      484
Scombroid toxin                              386
Norovirus                                    331
Bacillus cereus                              299
Campylobacter jejuni                         279
Ciguatoxin                                   267
Bacillus cereus; Clostridium perfringens     220
Chemical or toxin                            156
Shigella sonnei                              130
Vibrio parahaemolyticus                      120
Bacterium                                    114
Virus                                        101
Campylobacter unknown                         90
Hepatitis A                                   87
Name

### 4.2 Manual mapping — taxonomic corrections

Fuzzy matching cannot resolve these cases without biological knowledge:
- All Norovirus genogroup variants → `Norovirus` (same epidemiological unit)
- `Salmonella unknown` vs `Salmonella enterica` — kept separate (species matters for severity)
- `E. coli` subtypes (EPEC, STEC) → `Escherichia coli` for EDA-level aggregation
- Co-infections (e.g., `Bacillus cereus; Clostridium perfringens`) → primary pathogen

In [24]:
SPECIES_MANUAL = {
    # Norovirus — all genogroup variants are the same epidemiological entity
    "Norovirus genogroup I":                        "Norovirus",
    "Norovirus unknown":                            "Norovirus",
    "Norovirus genogroup I; Norovirus genogroup II": "Norovirus",

    # Salmonella — keep enterica separate from unknown for severity analysis
    "Salmonella unknown":                           "Salmonella",
    "Salmonella enterica; Salmonella enterica":     "Salmonella enterica",

    # E. coli subtypes → aggregate for EDA
    "Escherichia coli, Enteropathogenic":           "Escherichia coli",
    "Escherichia coli, Shiga toxin-producing":      "Escherichia coli",

    # Co-infections → primary pathogen
    "Bacillus cereus; Clostridium perfringens":     "Bacillus cereus",
    "Staphylococcus aureus; Bacillus cereus":       "Staphylococcus aureus",

    # Campylobacter normalisation
    "Campylobacter unknown":                        "Campylobacter",
    "Campylobacter":                                "Campylobacter",
}

print(f"Applying {len(SPECIES_MANUAL)} manual corrections...")
df = apply_manual_map(df, "Species_grouped", SPECIES_MANUAL)
print(f"Species_grouped unique after manual: {df['Species_grouped'].nunique():,}")
print()
df["Species_grouped"].value_counts().head(15)

Applying 11 manual corrections...
  11/11 keys matched and replaced
Species_grouped unique after manual: 145



Species_grouped
Unknown                    6406
Norovirus                  5276
Salmonella enterica        2348
Clostridium perfringens     732
Staphylococcus aureus       610
Escherichia coli            523
Bacillus cereus             519
Scombroid toxin             386
Campylobacter jejuni        279
Ciguatoxin                  267
Chemical or toxin           156
Shigella sonnei             130
Vibrio parahaemolyticus     120
Bacterium                   114
Campylobacter               108
Name: count, dtype: int64


### 4.3 Repeated-entry normalisation

RapidFuzz and manual mapping handle variant spellings, but cannot resolve
entries where the same species name was concatenated multiple times
(e.g., `"Salmonella enterica; Salmonella enterica; ..."` × up to 13 repetitions).
These arise from data entry errors in the original CDC records.

**Rule:** split on `;`, remove duplicate parts while preserving order,
rejoin. Genuine co-infections with two *different* pathogens
(e.g., `"Bacillus cereus; Norovirus"`) are left unchanged.

In [25]:
def clean_repeated_species(s: str) -> str:
    """
    Collapse repeated semicolon-separated entries to a single canonical form.

    Examples
    --------
    "Salmonella enterica; Salmonella enterica; Salmonella enterica"
        → "Salmonella enterica"

    "Bacillus cereus; Norovirus"          (genuine co-infection)
        → "Bacillus cereus; Norovirus"    (unchanged)
    """
    parts = [p.strip() for p in str(s).split(";")]
    unique_parts = list(dict.fromkeys(parts))   # deduplicate, preserve order
    return unique_parts[0] if len(unique_parts) == 1 else "; ".join(unique_parts)


before = df["Species_grouped"].nunique()
df["Species_grouped"] = df["Species_grouped"].apply(clean_repeated_species)
after  = df["Species_grouped"].nunique()

print(f"Species_grouped unique: {before} → {after}")
print()
print("Top 15 after cleaning:")
print(df[df["Species_grouped"] != "Unknown"]["Species_grouped"]
      .value_counts().head(15).to_string())

Species_grouped unique: 145 → 139

Top 15 after cleaning:
Species_grouped
Norovirus                  5276
Salmonella enterica        2362
Clostridium perfringens     734
Staphylococcus aureus       610
Escherichia coli            523
Bacillus cereus             519
Scombroid toxin             386
Campylobacter jejuni        279
Ciguatoxin                  267
Chemical or toxin           156
Shigella sonnei             130
Vibrio parahaemolyticus     120
Bacterium                   114
Campylobacter               108
Virus                       101


### 4.4 Species grouping result

In [26]:
species_before = df["Species"].nunique()
species_after  = df["Species_grouped"].nunique()
print(f"Species: {species_before:,} unique strings → {species_after:,} grouped categories")
print(f"Reduction: {(1 - species_after/species_before)*100:.0f}%")
print()
print("Top 10 pathogen groups by outbreak frequency:")
print(df[df["Species_grouped"] != "Unknown"]["Species_grouped"]
      .value_counts().head(10).to_string())

Species: 201 unique strings → 139 grouped categories
Reduction: 31%

Top 10 pathogen groups by outbreak frequency:
Species_grouped
Norovirus                  5276
Salmonella enterica        2362
Clostridium perfringens     734
Staphylococcus aureus       610
Escherichia coli            523
Bacillus cereus             519
Scombroid toxin             386
Campylobacter jejuni        279
Ciguatoxin                  267
Chemical or toxin           156


## 5. Location normalisation

### 5.1 The multi-venue problem

Unlike `Food` and `Species`, `Location` values are already standardised vocabulary —
no spelling variants or fuzzy matching needed. The sole problem is that outbreaks
spanning multiple venues (e.g., a catering company supplying both a restaurant and a
school) are recorded as semicolon-separated strings.

These 161 unique strings collapse to ~21 meaningful venue types once the primary
venue is extracted. The first-listed venue is used as the primary setting — this
reflects standard CDC reporting convention where the primary exposure site is listed
first.

In [27]:
# ── Audit before normalisation ───────────────────────────────
loc_vc = df['Location'].value_counts()
n_compound = df['Location'].str.contains(';', na=False).sum()

print(f"Location unique raw strings: {df['Location'].nunique()}")
print(f"Multi-venue records:         {n_compound:,} ({n_compound/len(df)*100:.1f}%)")
print()
print("Top 10 most frequent raw Location values:")
print(loc_vc.head(10).to_string())

Location unique raw strings: 161
Multi-venue records:         1,119 (5.9%)

Top 10 most frequent raw Location values:
Location
Restaurant                            10190
Unknown                                2503
Private Home/Residence                 1673
Catering Service                       1088
Banquet Facility                        366
Fast Food Restaurant                    365
School/College/University               353
Grocery Store                           301
Restaurant; Private Home/Residence      205
Prison/Jail                             193


### 5.2 Primary venue extraction

**Decision rationale:** Extract the first-listed venue as `primary_location`.

This is the correct approach because:
- CDC convention lists the primary exposure site first
- Multi-venue strings represent outbreak spread, not ambiguity about where exposure occurred
- No meaningful venue information is lost — the original `Location` column is preserved
- No fuzzy matching needed — the ~21 venue types use consistent spelling across all records

**What is NOT merged (intentional):**
- `Fast Food Restaurant` ≠ `Restaurant` — different HACCP requirements, menu structure,
  and typical failure modes (e.g., frozen patty temperature vs. fresh preparation)
- `Restaurant Buffet` ≠ `Restaurant` — buffets have distinct cross-contamination risk
  profiles and different food-holding time/temperature requirements
- `Nursing Home/Assisted Living Facility` kept as-is — regulatory category distinct
  from `Hospital` despite both serving vulnerable populations

In [28]:
# ── Primary venue extraction ─────────────────────────────────
df["primary_location"] = df["Location"].str.split(";").str[0].str.strip()

before = df["Location"].nunique()
after  = df["primary_location"].nunique()
print(f"Location: {before} raw unique strings → {after} primary venue types")
print(f"Reduction: {(1 - after/before)*100:.0f}%")
print()
print("primary_location distribution:")
print(df["primary_location"].value_counts().to_string())

Location: 161 raw unique strings → 21 primary venue types
Reduction: 87%

primary_location distribution:
primary_location
Restaurant                               10835
Unknown                                   2503
Private Home/Residence                    1980
Catering Service                          1139
Fast Food Restaurant                       433
Banquet Facility                           391
School/College/University                  357
Grocery Store                              304
Prison/Jail                                193
Nursing Home/Assisted Living Facility      186
Camp                                       119
Religious Facility                         110
Office/Indoor Workplace                    105
Fair/Festival                               83
Hospital                                    42
Child Daycare                               29
Farm/Dairy                                   8
Restaurant Buffet                            8
Other (Describe In Remarks)     

### 5.3 Validation and scope note

**What this achieves:** All Location-based analyses in EDA can now aggregate
across the 21 meaningful venue types without the noise of 161 compound strings.

**Scope limitation:** `Unknown` (13.3%) and multi-venue records (~5.9% of data) mean
that Location-based analyses cover approximately 81% of records with a clearly
identified single primary venue. This is stated explicitly in EDA analysis cells.

**Analytical implication:** The `primary_location` column is used in `03_EDA.ipynb`
for all setting-level analyses (outbreak frequency by venue, severity profiling,
setting × pathogen matrix). The original `Location` column is retained for records
where the compound string itself is analytically meaningful (e.g., identifying
multi-source outbreak chains).

In [29]:
# ── Verification ─────────────────────────────────────────────
# Check no new nulls introduced
nulls = df["primary_location"].isna().sum()
print(f"Null values in primary_location: {nulls}  (expected: 0)")

# Confirm 'Unknown' is preserved correctly
unknown_n = (df["primary_location"] == "Unknown").sum()
original_unknown = (df["Location"].str.split(";").str[0].str.strip() == "Unknown").sum()
print(f"Unknown records: {unknown_n:,}  (original: {original_unknown:,})")

# Show before/after for compound strings
sample_compound = df[df["Location"].str.contains(";", na=False)][
    ["Location", "primary_location"]
].drop_duplicates().head(8)
print()
print("Sample: compound Location → primary_location")
print(sample_compound.to_string(index=False))

Null values in primary_location: 0  (expected: 0)
Unknown records: 2,503  (original: 2,503)

Sample: compound Location → primary_location
                                     Location          primary_location
                 Restaurant; Catering Service                Restaurant
                    Restaurant; Grocery Store                Restaurant
           Restaurant; Private Home/Residence                Restaurant
   Private Home/Residence; Religious Facility    Private Home/Residence
        Private Home/Residence; Grocery Store    Private Home/Residence
     Private Home/Residence; Catering Service    Private Home/Residence
  Catering Service; School/College/University          Catering Service
School/College/University; Religious Facility School/College/University


## 6. Validation — no data loss

In [30]:
# Row count must be unchanged
assert len(df) == 18828, f"Row count changed: {len(df)}"
print(f"✓ Row count unchanged: {len(df):,}")

# No nulls in any grouped column
for col in ["Food_grouped", "Species_grouped", "primary_location"]:
    assert df[col].isna().sum() == 0, f"NaN in {col}"
print("✓ No missing values in Food_grouped, Species_grouped, primary_location")

# primary_location should have ~20 unique values (not 161)
n_primary = df["primary_location"].nunique()
assert n_primary < 30, f"primary_location has {n_primary} unique values — check extraction"
print(f"✓ primary_location: {n_primary} unique venue types (reduced from {df['Location'].nunique()})")

# Summary
print()
print("Final column summary:")
print(f"  Food:            {df['Food'].nunique():,} unique  →  "
      f"Food_grouped: {df['Food_grouped'].nunique():,}")
print(f"  Species:         {df['Species'].nunique():,} unique  →  "
      f"Species_grouped: {df['Species_grouped'].nunique():,}")
print(f"  Location:        {df['Location'].nunique():,} unique  →  "
      f"primary_location: {df['primary_location'].nunique():,}")
print()
df[["Location", "primary_location", "Food", "Food_grouped",
    "Species", "Species_grouped"]].head(5)

✓ Row count unchanged: 18,828
✓ No missing values in Food_grouped, Species_grouped, primary_location
✓ primary_location: 21 unique venue types (reduced from 161)

Final column summary:
  Food:            3,128 unique  →  Food_grouped: 1,366
  Species:         201 unique  →  Species_grouped: 139
  Location:        161 unique  →  primary_location: 21



,Location,primary_location,Food,Food_grouped,Species,Species_grouped
0,Restaurant,Restaurant,Unknown,Unknown,Unknown,Unknown
1,Unknown,Unknown,Custard,Custard,Unknown,Unknown
2,Restaurant,Restaurant,Unknown,Unknown,Unknown,Unknown
3,Restaurant,Restaurant,"Fish, Ahi",Fish,Scombroid toxin,Scombroid toxin
4,Private Home/Residence,Private Home/Residence,"Lasagna, Unspecified; Eggs, Other","Lasagna, Unspecified; Eggs, Other",Salmonella enterica,Salmonella enterica


## 7. Export

In [31]:
df.to_csv("cleaned_data_grouped.csv", index=False)
print("Saved: cleaned_data_grouped.csv")
print(f"  → {df.shape[0]:,} records × {df.shape[1]} columns")
print(f"  → Columns: {df.columns.tolist()}")
print()
print("New columns added:")
print("  Food_grouped      — food vehicle categories (3,128 → ~1,366)")
print("  Species_grouped   — pathogen categories (201 → ~145)")
print("  primary_location  — primary outbreak venue (161 → 21)")
print()
print("Next step: 03_EDA.ipynb")

Saved: cleaned_data_grouped.csv
  → 18,828 records × 12 columns
  → Columns: ['Year', 'Month', 'State', 'Location', 'Food', 'Species', 'Illnesses', 'Hospitalizations', 'Fatalities', 'Food_grouped', 'Species_grouped', 'primary_location']

New columns added:
  Food_grouped      — food vehicle categories (3,128 → ~1,366)
  Species_grouped   — pathogen categories (201 → ~145)
  primary_location  — primary outbreak venue (161 → 21)

Next step: 03_EDA.ipynb
